# AllRecipes Data Analysis Project

## 1. Introduction
*(We will introduce the data set and our guiding question here.)*

## 2. Data Cleaning and Summary

### 2.1 Imports

In [126]:
import pandas as pd
import numpy as np
from scipy import stats

### 2.2 Loading the datasets

In [127]:
url_recipes = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/all_recipes.csv'
url_cuisines = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-16/cuisines.csv'

# Turn links into DataFrames
df_recipes = pd.read_csv(url_recipes)
df_cuisines = pd.read_csv(url_cuisines)

df_cuisines.head()

,name,country,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Saganaki (Flaming Greek Cheese),Greek,https://www.allrecipes.com/recipe/263750/flami...,John Mitzewich,2024-02-07,"1 (4 ounce) package kasseri cheese, 1 tablespo...",391.0,25.0,15.0,16.0,4.8,25.0,22.0,10,5,15,2.0
1,Coney Island Knishes,Jewish,https://www.allrecipes.com/recipe/272334/coney...,John Mitzewich,2024-11-26,"2 ¾ cups all-purpose flour, or more as needed,...",301.0,17.0,31.0,7.0,4.6,10.0,9.0,30,75,180,16.0
2,Diana's Hawaiian Bread Rolls,Australian and New Zealander,https://www.allrecipes.com/recipe/22797/dianas...,CHIPPENDALE,2022-07-14,1 ½ cups warm water (110 degrees F/45 degrees ...,64.0,3.0,9.0,1.0,4.3,126.0,104.0,20,15,180,12.0
3,Chilean Pebre,Chilean,https://www.allrecipes.com/recipe/273763/chile...,Heidi,2025-01-31,"½ cup chopped cilantro, ¼ cup olive oil, ¼ cup...",106.0,9.0,7.0,1.0,5.0,1.0,1.0,10,0,10,6.0
4,Tex-Mex Sheet Cake,Tex-Mex,https://www.allrecipes.com/recipe/22388/tex-me...,Ann,2025-02-18,"2 cups all-purpose flour, 1 ½ cups brown sugar...",449.0,23.0,58.0,7.0,3.8,13.0,11.0,30,15,45,15.0


In [128]:
df_recipes.head()

,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0


### 2.3 Data cleaning

In [129]:
# Use outer join to merge cuisine labels onto recipes
df = pd.merge(df_recipes, 
              df_cuisines[['url', 'country']].rename(columns={'country': 'cuisine'}), 
              on='url', 
              how='outer')

# Standardize column types to numeric
num_cols = ["calories", "fat", "carbs", "protein", "avg_rating", 
            "total_ratings", "reviews", "prep_time", "cook_time", 
            "total_time", "servings"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows missing essential metrics
df.dropna(subset=['name', 'avg_rating', 'ingredients', 'prep_time', 
                  'cook_time', 'total_time'], inplace=True)

# Remove recipes with 0 time
df = df[df['total_time'] > 0]

Other than empty ratings, time and ingredients using NA, some values in the 'author' column are 'Allrecipes Member' which may be the NA values of authors' names but there is no need to remove those recipes as we are not concerned with the authors of the recipes.

### 2.4 Some feature engineering for summary statistics

In [130]:
# Count ingredient lines
def count_ingredients(s):
    if pd.isna(s): return 0
    return len([item for item in str(s).split(",") if item.strip()])

df['n_ingredient_lines'] = df['ingredients'].apply(count_ingredients)

In [131]:
df['log_total_time'] = np.log1p(df['total_time']) # log-transform heavily skewed data
df_filtered = df[df['total_ratings'] >= 10].copy() # Filter for reliable averages

# Actionable Score
df_filtered['z_rating'] = stats.zscore(df_filtered['avg_rating'])

df_filtered['z_ingredients'] = stats.zscore(df_filtered['n_ingredient_lines'])
df_filtered['complexity_score'] = df_filtered['z_rating'] - df_filtered['z_ingredients']

df_filtered['z_log_time'] = stats.zscore(np.log1p(df_filtered['total_time']))
df_filtered['action_score'] = df_filtered['z_rating'] - df_filtered['z_log_time']

df_filtered['ultimate_score'] = df_filtered['z_rating'] - (df_filtered['z_log_time'] + df_filtered['z_ingredients'])

### 2.5 Key summary statistics

In [132]:
print("--- General Dataset Shape ---")
print(f"Total recipes after cleaning: {len(df):,}")
print(f"Recipes with cuisine tags: {df['cuisine'].notna().sum():,}")

print("\n" + "="*100)

print("\n--- Key Factors Summary ---")
rating_stats = df[['avg_rating', 'total_ratings', 'total_time', 'n_ingredient_lines']].describe().loc[['min', '25%', 'mean', '50%', '75%', 'max']]
display(rating_stats)

print("\n" + "="*100)

# Do faster recipes get more reviews?
print("\n--- Total Reviews by Time Category ---")
df['time_quartile'] = pd.qcut(df['total_time'], 4, labels=['Very Fast', 'Fast', 'Medium', 'Slow'])
engagement = df.groupby('time_quartile', observed=True)['total_ratings'].agg(['mean', 'std'])
display(engagement)

print("\n" + "="*100)

# Is there a correlation between Time and Rating?
corr_time_rating = df[['total_time', 'avg_rating']].corr().iloc[0,1]
print(f"\nPearson Correlation between Total Time and Average Rating: {corr_time_rating:.4f}")

# What are the fastest recipes with high ratings?
print("\n--- Top 5 Fastest High-Rated Recipes (High Rating, Low Time) ---")
display(df_filtered.sort_values('action_score', ascending=False)[['name', 'total_time', 'avg_rating', 'n_ingredient_lines', 'action_score']].head(5))

print("\n" + "="*100)

# Is there a correlation between Number of Ingredients and Rating?
corr_ingredients_rating = df[['n_ingredient_lines', 'avg_rating']].corr().iloc[0,1]
print(f"\nPearson Correlation between Number of Ingredients and Average Rating: {corr_ingredients_rating:.4f}")

# What are the low-ingredient recipes with high ratings?
print("--- Top 5: Simplest High-Rated Recipes (Low Ingredients) ---")
display(df_filtered.sort_values('complexity_score', ascending=False)
        [['name', 'avg_rating', 'n_ingredient_lines', 'complexity_score']].head(5))

print("\n" + "="*100)

# Is there a correlation between Time and Number of Ingredients?
complexity_corr = df[['total_time', 'n_ingredient_lines']].corr().iloc[0,1]
print(f"\nCorrelation between Total Time and Number of Ingredients: {complexity_corr:.4f}")

# What are the fastest, low-ingredient recipes with high ratings?
print("\n--- Top 5: Simplest and Fastest High-Rated Recipes (High Rating, Low Time, Low Ingredients) ---")
display(df_filtered.sort_values('ultimate_score', ascending=False)
        [['name', 'avg_rating', 'total_time', 'n_ingredient_lines', 'ultimate_score']].head(5))

# ------------------------------------------------------------------------------------------------------------
# Initial findings for actionable recipes

# Define actionable recipes as as the top 10% of ultimate_score
top_threshold = df_filtered['ultimate_score'].quantile(0.9)
df_filtered['is_top_actionable'] = df_filtered['ultimate_score'] >= top_threshold

print("\n" + "="*100)

# Do actionable recipes have more sugar/fat (calories)?
print("--- Nutritional Comparison: Actionable vs. Others ---")
nutritional_comp = df_filtered.groupby('is_top_actionable')[['calories', 'fat', 'protein']].median()
display(nutritional_comp)

--- General Dataset Shape ---
Total recipes after cleaning: 13,104
Recipes with cuisine tags: 1,001


--- Key Factors Summary ---


,avg_rating,total_ratings,total_time,n_ingredient_lines
min,1.000000,1.000000,1.000000,1.000000
25%,4.400000,5.000000,30.000000,8.000000
mean,4.531219,102.082036,149.599435,12.390644
50%,4.600000,26.000000,55.000000,11.000000
75%,4.800000,111.000000,105.000000,16.000000
max,5.000000,997.000000,60485.000000,46.000000




--- Total Reviews by Time Category ---


,mean,std
time_quartile,,
Very Fast,100.252870,172.070778
Fast,103.902074,176.484700
Medium,110.125816,181.118917
Slow,93.493643,162.234260




Pearson Correlation between Total Time and Average Rating: 0.0080

--- Top 5 Fastest High-Rated Recipes (High Rating, Low Time) ---


,name,total_time,avg_rating,n_ingredient_lines,action_score
8077,Christmas Poinsettia Punch,5.0,5.0,9,4.022274
1761,Peach Cosmo,5.0,5.0,5,4.022274
7213,Prickly Pear Margarita,5.0,5.0,6,4.022274
2227,Simple Cranberry Spinach Salad,5.0,5.0,5,4.022274
4417,Pumpkin Pie Spice I,1.0,4.7,4,3.979555


... (2 outputs left out)



Correlation between Total Time and Number of Ingredients: 0.0048

--- Top 5: Simplest and Fastest High-Rated Recipes (High Rating, Low Time, Low Ingredients) ---


,name,avg_rating,total_time,n_ingredient_lines,ultimate_score
14909,Classic White Wine Spritzer,4.9,5.0,2,5.483654
4417,Pumpkin Pie Spice I,4.7,1.0,4,5.442965
1761,Peach Cosmo,5.0,5.0,5,5.301106
2227,Simple Cranberry Spinach Salad,5.0,5.0,5,5.301106
5676,Classic Ham Glaze,4.9,5.0,3,5.299076



--- Nutritional Comparison: Actionable vs. Others ---


,calories,fat,protein
is_top_actionable,,,
False,317.0,15.0,9.0
True,181.0,9.0,4.0


Insights about the data is that the mean (4.53) and median (4.60) for avg_ratings are very high and close to each other. This suggests that the dataset has a large positivity bias for ratings. On Allrecipes, a 4.0 rating is not actually good comparatively, but it is actually below average. Hence, this statistic shows that the bar for a good rating is actually very high and that the most actionable recipes should only contain recipes that are above the median (4.60). 

Moreover, there is a large difference between mean (102) and median (26) for total_ratings as there are many ratings (max=997) that are pulling the average up. This suggests that the data for total ratings is largely right skewed. Hence, recipes with less than 10 reviews are filtered out to ensure that the actionable recipes found are reliable and not just based on one review. This ensures that the actionable recipes found are not only are fast, have high ratings and a low number of ingredients, but they are also reliable with at least more than 10 number of reviews.

Additionally, there is also a large difference between mean (150) and median (55) for total_time. Similar to total_ratings, the data for total_time is also largely right skewed. Hence, a log_total_time column is made for a more meaningful visualisation.

Another interesting insight about the data is that the time category with the highest mean total ratings (110 ratings) is the medium or third quartile of the total time for all the recipes, while the slow category or the last quartile has the lowest (93). This suggests that time is a real factor that affects the user's decision on which recipe to follow. A medium time is preferred by the community as seen by the mean total reviews which may mean that the most actionable recipes determined by current readers are those that take medium time, followed by fast, very fast and slow time. This validates our guiding question as it shows that the total time taken for a recipe plays a role in the willingness of users to follow the recipe. 

Moving on to investigate the relationship between rating with time and number of ingredients, there is lack of correlation between time, ingredients and ratings (r = 0.008 between time and rating, r = 0.0747 between ingredients and rating, and r = 0.0048 between ingredients and time). This shows that there is no correlation between ratings with time and with ingredients, making the search for actionable recipes valid. 

The most actionable recipe based on time and both time and number of ingredients are drinks and the top 5 actionable recipes is found to contain many drinks. However, the most actionable recipe based on the number of ingredients is food and the top 5 contains mainly food, soup and creams. This suggests that for high ratings, the types of recipes for lowest time and lowest number of ingredients differ slightly. However, the most actionable recipes are often components (creams) or drinks. This suggests that recipes that take less time are more likely recipes for drinks. This validates our finding that recipes with medium time have the highest mean total reviews as the medium time category may have more recipes for food that take less time. Hence, current users may be more attracted to these recipes.

The fact that actionable recipes have nearly half the calories (181 vs 317) and significantly less fat is a major insight. It suggests that actionable recipes are fast, lighter and potentially healthier. However, this may also be due to the fact that the most actionable recipes are mostly drinks that would usually have less calories, fat and protein compared to food. 

## 3. Visualisation
- Scatter plot: Rating vs. Total Time (using color to separate food and drinks? -> if cannot do this, just note down somewhere that drinks on average take less time)
    - x = total_time, y = avg_rating, one point per recipe
    - With 13,000+ recipes, a standard scatter plot will suffer from overplotting.
    - Improvement: Use Hexbinning or Density Contours. Instead of individual points,
    use a hexbin plot where the color intensity represents the number of recipes.
    - Improvement: Instead of showing all points, draw a line connecting the highest 
    rated recipe at each time interval (Any recipe on or near that line is "The Best 
    of its Class.")
    - The "Actionable" Layer: Overlay a red dashed rectangle in the "High Rating
    (>4.5) / Low Time (<30 mins)" quadrant. This visually defines "Actionable" for
    the reader immediately.
    - Possible: use a Marginal Plot (histograms on the top and side of the scatter 
    plot) to show where the volume of recipes truly sits.
- Box plot: Rating distribution across time buckets
    - Bin total_time into groups (e.g. under 30 min, 30–60 min, 1–2 hrs, 2+ hrs)
    - Show the spread of avg_rating within each bucket
    - Improvement: Add Sample Sizes (N=) above each box. If the "2+ hrs" box is
    based on 200 recipes but the "Under 30 min" box is based on 5,000, your reader
    needs to know the latter is a more "reliable" finding.
    - Improvement: There may be a lot of outliers. Set showfliers=False in Seaborn 
    to hide individual outliers, or use a Violin Plot with a box plot inside it
    - Visual: Use a Notched Box Plot / Violin plot. If the notches of two boxes do
    not overlap, it’s a visual shorthand for saying the difference in their medians
    is statistically significant.
    - Refined Buckets: Instead of quartiles, use "Human" buckets: 0-15m (Ultra-
    Fast), 15-45m (Standard), 45-90m (Slow), 90m+ (Event). This is more relatable to 
    a home cook.
- Heatmap: Time and Ingredients list against rating
    - X-axis: Total Time Bins (e.g., 0-15, 15-30, 30-60, 60+)
    - Y-axis: Ingredient Count Bins (e.g., 1-5, 6-10, 11-15, 16+)
    - Color/Intensity: The Average Rating (or the percentage of recipes that are "5-
    stars" for better color differentiation)
    - The cell in the top-left (Low Time + Low Ingredients) will show a specific color. If that color is "Dark Green" (High Rating), you have visual, empirical proof that simplicity is rewarded.
    - It might show that as you move to the bottom-right (High Time + High Ingredients), the ratings don't actually get higher, proving they aren't "more actionable."

## 4. Discussions
*(We will explain the insights from our plots here.)*

## 5. Teamwork
*(Tasks allocation paragraph goes here.)*


## 6. References
* TidyTuesday AllRecipes Dataset: https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-09-16/readme.md